# 06 - Blood Pressure Regression with MLflow

Train Linear Regression, Random Forest, and Gradient Boosting regressors for systolic and diastolic blood pressure prediction.

In [ ]:
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor

try:
    dbutils.widgets.text('experiment_name', '/Shared/CardioTwin_AI_BP_Regression')
    dbutils.widgets.text('minimum_quality_score', '0.40')
    experiment_name = dbutils.widgets.get('experiment_name')
    minimum_quality_score = float(dbutils.widgets.get('minimum_quality_score'))
except Exception:
    experiment_name = 'CardioTwin_AI_BP_Regression'
    minimum_quality_score = 0.40

features_df = spark.table('cardio_ppg_features')
quality_df = spark.table('cardio_signal_quality').select('window_id', 'signal_quality_score')
model_df = (
    features_df.join(quality_df, on='window_id', how='left')
    .fillna({'signal_quality_score': 0.0})
    .filter(F.col('signal_quality_score') >= minimum_quality_score)
)

pdf = model_df.toPandas()
feature_cols = [
    'mean_amplitude', 'std_amplitude', 'min_amplitude', 'max_amplitude', 'amplitude_range',
    'peak_count', 'heart_rate_approx', 'signal_energy', 'mean_abs_change',
    'systolic_proxy', 'diastolic_proxy', 'pulse_pressure_proxy', 'signal_quality_score'
]
target_cols = ['systolic_bp', 'diastolic_bp']

X = pdf[feature_cols].fillna(0.0)
y = pdf[target_cols].astype(float)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest Regressor': RandomForestRegressor(n_estimators=150, random_state=42),
    'Gradient Boosting Regressor': MultiOutputRegressor(GradientBoostingRegressor(random_state=42)),
}

mlflow.set_experiment(experiment_name)
leaderboard = []
trained_models = {}

for model_name, model in models.items():
    with mlflow.start_run(run_name=model_name) as run:
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)

        metrics = {
            'mae': float(mean_absolute_error(y_test, predictions)),
            'rmse': float(np.sqrt(mean_squared_error(y_test, predictions))),
            'r2_score': float(r2_score(y_test, predictions, multioutput='variance_weighted')),
            'mae_systolic': float(mean_absolute_error(y_test['systolic_bp'], predictions[:, 0])),
            'mae_diastolic': float(mean_absolute_error(y_test['diastolic_bp'], predictions[:, 1])),
        }

        mlflow.log_param('model_name', model_name)
        mlflow.log_param('feature_count', len(feature_cols))
        mlflow.log_param('targets', ','.join(target_cols))
        mlflow.log_param('minimum_quality_score', minimum_quality_score)
        mlflow.log_metrics(metrics)

        plot_path = f'/tmp/{model_name.lower().replace(" ", "_")}_bp_predictions.png'
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        for index, target in enumerate(target_cols):
            axes[index].scatter(y_test[target], predictions[:, index], color='#2b8cbe', alpha=0.85)
            min_value = min(float(y_test[target].min()), float(predictions[:, index].min()))
            max_value = max(float(y_test[target].max()), float(predictions[:, index].max()))
            axes[index].plot([min_value, max_value], [min_value, max_value], color='#333333', linestyle='--')
            axes[index].set_title(target)
            axes[index].set_xlabel('Actual')
            axes[index].set_ylabel('Predicted')
        plt.tight_layout()
        plt.savefig(plot_path, dpi=160)
        plt.close()
        mlflow.log_artifact(plot_path)

        mlflow.sklearn.log_model(model, artifact_path='model')
        leaderboard.append({'run_id': run.info.run_id, 'model_name': model_name, **metrics})
        trained_models[model_name] = model

leaderboard_pdf = pd.DataFrame(leaderboard).sort_values('rmse', ascending=True)
display(leaderboard_pdf)

best_model_name = leaderboard_pdf.iloc[0]['model_name']
best_model = trained_models[best_model_name]
best_model_path = '/tmp/cardio_best_bp_regression_model'
mlflow.sklearn.save_model(best_model, best_model_path)

print(f'Best BP regression model: {best_model_name}')
print(f'Local MLflow model saved at: {best_model_path}')
